In [4]:
import pandas as pd 
import numpy as np 


path='/Users/suecortesmachado/Downloads/olist/'

orders = pd.read_csv(path + 'olist_orders_dataset.csv')
customers = pd.read_csv(path + 'olist_customers_dataset.csv')
order_items = pd.read_csv(path + 'olist_order_items_dataset.csv')
products = pd.read_csv(path + 'olist_products_dataset.csv')
payments = pd.read_csv(path + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(path + 'olist_order_reviews_dataset.csv')
sellers = pd.read_csv(path + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(path + 'product_category_name_translation.csv')


print('All files loaded successfully!')
print(f'Orders: {len(orders)} rows')

All files loaded successfully!
Orders: 99441 rows


In [5]:
for name, table in {'orders':orders, 'customers':customers,
                        'order_items':order_items, 'products':products}.items():
  print(f"--- {name} ---")
  print(f"Shape: {table.shape}")
  print(table.isnull().sum()[table.isnull().sum() > 0])
  print()

--- orders ---
Shape: (99441, 8)
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- customers ---
Shape: (99441, 5)
Series([], dtype: int64)

--- order_items ---
Shape: (112650, 7)
Series([], dtype: int64)

--- products ---
Shape: (32951, 9)
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64



In [ ]:
Exercise 1 — sales overview
What is the total number of orders, total revenue, and average order value? Use the order_items table.
Revenue = sum of price + freight_value.

In [9]:
print(order_items.columns)

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')


In [17]:
order_items['revenue']= order_items['price']+order_items['freight_value']

total_orders= order_items['order_id'].nunique()
total_revenue= order_items['revenue'].sum()
avg_order_value= order_items.groupby('order_id')['revenue'].sum().mean()

print(f'Total orders: {total_orders}')
print(f'Total revenue: {total_revenue:.2f}')
print(f'Average order value: {avg_order_value:.2f}')

Total orders: 98666
Total revenue: 15843553.24
Average order value: 160.58


In [ ]:
Exercise 2 — top states
Which Brazilian states have the most customers? Show the top 10 states by number of unique customers, 
sorted descending. Use the customers table.

In [19]:
print(customers.columns)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')


In [22]:
top_states= (customers.groupby('customer_state')['customer_id'].nunique().sort_values(ascending=False).head(10)
)
print(top_states)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: customer_id, dtype: int64


In [ ]:
Exercise 3 — revenue by category
What are the top 10 product categories by total revenue? You need to merge: order_items + products + category_translation. 
Show category name in English and total revenue sorted highest first.

In [23]:
print(order_items.columns)
print(products.columns)
print(category_translation.columns)

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'revenue'],
      dtype='object')
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')
Index(['product_category_name', 'product_category_name_english'], dtype='object')


In [27]:
df = (order_items
      .merge(products, on='product_id', how='left')      # add product info
      .merge(category_translation, on='product_category_name', how='left'))     # add english category names


top_categories = (df
                  .groupby('product_category_name')['revenue']
                  .sum()
                  .sort_values(ascending=False )
                  .head(10))

print(top_categories)

product_category_name
beleza_saude              1441248.07
relogios_presentes        1305541.61
cama_mesa_banho           1241681.72
esporte_lazer             1156656.48
informatica_acessorios    1059272.40
moveis_decoracao           902511.79
utilidades_domesticas      778397.77
cool_stuff                 719329.95
automotivo                 685384.32
ferramentas_jardim         584219.21
Name: revenue, dtype: float64


In [ ]:
Exercise 4 — delivery performance
Calculate the average delivery time in days per state. 
Delivery time = difference between order_delivered_customer_date and order_purchase_timestamp. 
Which state has the fastest and slowest delivery?


In [28]:
print(orders.columns)
print(customers.columns)
print(order_items.columns)
print(products.columns)
print(payments.columns)
print(reviews.columns)
print(sellers.columns)
print(category_translation.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'revenue'],
      dtype='object')
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 're

In [31]:
df = orders.merge(customers, on='customer_id', how='left')


df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])


df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days


delivery_by_state = (df
                     .groupby('customer_state')['delivery_days']
                     .mean()
                     .sort_values(ascending=True)
                     .round(1))

print("Fastest states:")
print(delivery_by_state.head(5))
print("\nSlowest states:")
print(delivery_by_state.tail(5))

Fastest states:
customer_state
SP     8.3
PR    11.5
MG    11.5
DF    12.5
SC    14.5
Name: delivery_days, dtype: float64

Slowest states:
customer_state
PA    23.3
AL    24.0
AM    26.0
AP    26.7
RR    29.0
Name: delivery_days, dtype: float64


In [ ]:
Exercise 5 — payment methods
What percentage of orders use each payment type (credit card, boleto, etc.)? 
Show payment_type and its percentage of total orders, sorted by most popular.


In [32]:
print(payments.columns)

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')


In [35]:
payment_counts= payments['payment_type'].value_counts()
payment_percentage= (payment_counts/payment_counts.sum()*100).round(1)
payment_percentage= payment_percentage.to_frame(name='percentage')
print(payment_percentage)


              percentage
payment_type            
credit_card         73.9
boleto              19.0
voucher              5.6
debit_card           1.5
not_defined          0.0


In [ ]:
Exercise 6 — review score analysis
Find the average review score per product category. 
Then find which categories have the worst reviews (below 3.5 average). 
Merge: reviews + orders + order_items + products + category_translation.

In [40]:
print(reviews.columns)
print(orders.columns)
print(order_items.columns)
print(products.columns)
print(category_translation.columns)

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'revenue'],
      dtype='object')
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')
Index(['product_category_name', 'product_category_name_english'], dtype='object')


In [52]:
df=(reviews
    .merge(orders, on='order_id', how= 'left')
    .merge(order_items, on='order_id',how= 'left')
    .merge(products, on='product_id', how='left')
    .merge(category_translation, on='product_category_name')
)

Avg_review = df.groupby('product_category_name_english')['review_score'].mean().round(2).sort_values(ascending=True)

worst_category= Avg_review[Avg_review<3.5]

print("Categories with average review below 3.5:")
print(worst_category)






Categories with average review below 3.5:
product_category_name_english
security_and_services    2.50
diapers_and_hygiene      3.26
office_furniture         3.49
Name: review_score, dtype: float64


In [ ]:
7 — monthly revenue trend
Show total revenue per month across all years.
Extract month and year from order_purchase_timestamp, then groupby and sum. 
Create a pivot table showing year as rows and month as columns.

In [54]:
print(orders.columns)

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')


In [57]:
df= order_items.merge(orders, on='order_id', how='left')

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['year'] = df['order_purchase_timestamp'].dt.year
df['month'] = df['order_purchase_timestamp'].dt.month 

df['revenue'] = df['price'] + df['freight_value']

revenue_pivot = df.pivot_table(
    values='revenue',
    index='year',
    columns='month',
    aggfunc='sum'
).round(2)

print(revenue_pivot)

month          1          2           3           4           5           6   \
year                                                                           
2016          NaN        NaN         NaN         NaN         NaN         NaN   
2017    137188.49  286280.62   432048.59   412422.24   586190.95   502963.04   
2018   1107301.89  986908.96  1155126.82  1159698.04  1149781.82  1022677.11   

month          7           8          9          10          11         12  
year                                                                        
2016          NaN         NaN     354.75   56808.84         NaN      19.62  
2017    584971.62   668204.60  720398.91  769312.37  1179143.77  863547.23  
2018   1058728.03  1003308.47     166.46        NaN         NaN        NaN  


In [ ]:
boss level Exercise 8 — seller performance
Build a complete seller performance report showing: total orders per seller, total revenue, average review score, and average delivery days. 
Classify each seller as 'Top', 'Average', or 'Poor' using np.select based on their review score. Sort by total revenue descending.

In [59]:
print(reviews.columns)
print(orders.columns)
print(sellers.columns)

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')


In [64]:
df = (order_items
      .merge(orders, on='order_id', how='left')
      .merge(reviews, on='order_id', how='left')
      .merge(sellers, on='seller_id', how='left'))


df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['revenue'] = df['price'] + df['freight_value']


seller_report = df.groupby('seller_id').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('revenue', 'sum'),
    avg_review=('review_score', 'mean'),
    avg_delivery_days=('delivery_days', 'mean')
).round(2)


conditions = [
    seller_report['avg_review'] > 4.0,
    seller_report['avg_review'] >= 3.0,
    seller_report['avg_review'] < 3.0
]
choices = ['Top', 'Average', 'Poor']
seller_report['performance'] = np.select(conditions, choices, default='Unknown')


seller_report = seller_report.sort_values('total_revenue', ascending=False)

print(seller_report.head(10))


                                  total_orders  total_revenue  avg_review  \
seller_id                                                                   
4869f7a5dfa277a7dca6462dcf3b52b2          1132      249640.70        4.12   
7c67e1448b00f6e969d365cea6b010ab           982      241374.82        3.35   
4a3ca9315b744ce9f8e9374361493884          1806      238440.31        3.80   
53243585a1d6dc2643021fd1853d8905           358      235856.68        4.08   
fa1c13f2614d7b5c4749cbc52fecda94           585      204084.73        4.34   
da8622b14eb17ae2831f4ac5b9dab84a          1314      188062.51        4.07   
7e93a43ef30c4f03f38b393420bc753a           336      182754.05        4.21   
1025f0e2d44d7041d6cf58b6550e0bfa           915      174710.82        3.85   
7a67c85e85bb2ce8582c35f2203ad736          1160      163278.78        4.23   
955fee9216a65b617aa5c0531780ce60          1287      160694.86        4.05   

                                  avg_delivery_days performance  
seller_id